# Simulating the Multi-Speed Collisionless QLBM

The multi-speed (`MSLattice`) discretization specifies the number of velocities *per dimension*
instead of naming a DdQq stencil, which lets each axis carry several speeds.

`qlbm` builds its circuits as `qarp` blocks and simulates them with `qarp`'s statevector simulator.
A run is described by a `SimulationConfig`, which ties the four algorithmic circuits — initial
conditions, time step algorithm, postprocessing, and measurement — to a compiler optimization level
and a shot budget. `cfg.validate()` type-checks the configuration and `cfg.prepare_for_simulation()`
lowers every component into a built `qarp` block. A `QarpRunner` then drives the time loop and writes
one ParaView frame per time step into `output_dir/paraview`.

`statevector_snapshots=True` is the fast path: the runner hands the state produced at step $k$ back to
the simulator as the *initial state* of step $k+1$, so simulating $n$ steps costs $n$ applications of
the algorithm instead of $\frac{n(n+1)}{2}$. Passing `qarp.EXACT` instead of a shot count returns exact
outcome probabilities rather than sampled counts.

> **Demo-sized.** The lattice below is an 8x8 domain with 4 velocities per axis and one
> bounce-back obstacle, simulated for 8 time steps. Simulation cost grows exponentially in the number of qubits, so scale the lattice and
> `NUM_STEPS` up deliberately.

In [ ]:
from qlbm.components import (
    MSQLBM,
    EmptyPrimitive,
    GridMeasurement,
    MSInitialConditions,
)
from qlbm.infra import QarpRunner, SimulationConfig
from qlbm.lattice import MSLattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = MSLattice(
    {
        "lattice": {
            "dim": {"x": 8, "y": 8},
            "velocities": {
                "x": 4,
                "y": 4,
            },
        },
        "geometry": [
            {"shape": "cuboid", "x": [5, 6], "y": [3, 5], "boundary": "bounceback"},
        ],
    }
)


output_dir = "qlbm-output/ms-4x4-8x8-1-obstacle-qarp"
create_directory_and_parents(output_dir)

In [ ]:
# Number of shots to sample for each time step
NUM_SHOTS = 2**12

# Number of time steps to simulate
NUM_STEPS = 8

In [ ]:
cfg = SimulationConfig(
    initial_conditions=MSInitialConditions(lattice),
    algorithm=MSQLBM(lattice, group_velocities=True),
    postprocessing=EmptyPrimitive(lattice),
    measurement=GridMeasurement(lattice),
    optimization_level=0,
    shots=NUM_SHOTS,
)

cfg.validate()
cfg.prepare_for_simulation()

In [ ]:
runner = QarpRunner(cfg, lattice, seed=1234)


# Snapshots carry the statevector between time steps, making the loop O(NUM_STEPS)
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)